# W3D3 - Distribution Analysis

Analyzing the distribution of numerical columns and interpreting how mean and median behave differently depending on skew.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Synthetic "employees" dataset - self-contained, no internet needed
rng = np.random.default_rng(seed=42)
n = 200

departments = rng.choice(
    ["Engineering", "Sales", "Marketing", "HR"], size=n, p=[0.4, 0.3, 0.2, 0.1]
)
years_experience = np.clip(rng.normal(6, 3.5, size=n), 0, 25).round(1)

# Base salary driven by department + experience, plus a few high outliers
base_salary = {
    "Engineering": 85000, "Sales": 65000, "Marketing": 60000, "HR": 58000,
}
salary = np.array([base_salary[d] for d in departments]) \
    + years_experience * 1800 \
    + rng.normal(0, 5000, size=n)
# Inject a handful of high-earner outliers to make mean/median diverge
outlier_idx = rng.choice(n, size=6, replace=False)
salary[outlier_idx] += rng.uniform(60000, 120000, size=6)
salary = salary.round(0)

satisfaction_score = np.clip(rng.normal(7, 1.5, size=n), 1, 10).round(1)

df = pd.DataFrame({
    "department": departments,
    "years_experience": years_experience,
    "salary": salary,
    "satisfaction_score": satisfaction_score,
})

df.head()

,department,years_experience,salary,satisfaction_score
0,Marketing,8.3,78741.0,4.5
1,Sales,4.6,74426.0,7.8
2,Marketing,6.0,73450.0,5.4
3,Sales,5.4,71197.0,7.4
4,Engineering,7.2,97062.0,4.9


## Distributions of each numeric column

In [2]:
numeric_cols = ["years_experience", "salary", "satisfaction_score"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, numeric_cols):
    sns.histplot(df[col], kde=True, ax=ax, color="#4c72b0")
    mean_val = df[col].mean()
    median_val = df[col].median()
    ax.axvline(mean_val, color="red", linestyle="--", label=f"Mean = {mean_val:.1f}")
    ax.axvline(median_val, color="green", linestyle="-", label=f"Median = {median_val:.1f}")
    ax.set_title(col)
    ax.legend(fontsize=8)

fig.suptitle("Distribution of Numeric Columns (Mean vs. Median)", fontsize=14)
fig.tight_layout()
fig.savefig("distributions_W3D3.png", dpi=150)
plt.show()

## Interpreting mean vs. median

In [3]:
for col in numeric_cols:
    mean_val = df[col].mean()
    median_val = df[col].median()
    diff = mean_val - median_val
    print(f"{col}: mean={mean_val:.2f}, median={median_val:.2f}, "
          f"difference={diff:+.2f}")

years_experience: mean=6.12, median=6.20, difference=-0.08
salary: mean=84782.05, median=81851.50, difference=+2930.55
satisfaction_score: mean=6.92, median=7.00, difference=-0.08


**Interpretation:**

- `years_experience` and `satisfaction_score` are roughly symmetric, so
  their mean and median are close together.
- `salary` has a handful of high-earner outliers, which pulls the **mean**
  upward while the **median** stays closer to what a "typical" employee
  earns. When mean is noticeably higher than median, that's a sign of
  right-skew (a few large values dragging the average up) — the median is
  usually the more representative "typical value" in that situation.